<a href="https://colab.research.google.com/github/vxkudire/python-basic-agents/blob/main/Hello_Agent_CSV_FAQ_Agent_(Stub_File).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-experimental langchain-openai pandas requests==2.32.4
!pip install gdown

In [ ]:
import pandas as pd
import os
import sys
import gdown
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# ==========================================
#  PART 1: AUTOMATIC FILE DOWNLOADER
# ==========================================

files_to_download = {
    "saas_docs.csv":         "https://drive.google.com/file/d/1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq/view?usp=sharing",
    "credit_card_terms.csv": "https://drive.google.com/file/d/1_giivc_B0urOKpct0XY2yVZuxW3Eenuf/view?usp=sharing",
    "hospital_policy.csv":   "https://drive.google.com/file/d/1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm/view?usp=sharing",
    "ecommerce_faqs.csv":    "https://drive.google.com/file/d/1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p/view?usp=sharing"
}

print("--- Downloading Files from Google Drive ---")
for filename, url in files_to_download.items():
    if not os.path.exists(filename):
        gdown.download(url, filename, quiet=False, fuzzy=True)
        print(f"Downloaded: {filename}")
    else:
        print(f"Skipped: {filename} (Already exists)")
print("--- Download Complete ---\n")


# ==========================================
#  PART 2: AI AGENT SETUP (MULTI-FILE)
# ==========================================

# 1. SETUP: Get API Key Securely
print("ENTER YOUR OPENAI API KEY BELOW:")
apikey = getpass()

# 2. LOAD ALL CSVs INTO A LIST
dataframes = [] # We will store all the loaded tables here
loaded_names = []

try:
    for filename in files_to_download.keys():
        df = pd.read_csv(filename)
        dataframes.append(df)
        loaded_names.append(filename)
        print(f"SUCCESS: Loaded '{filename}' ({len(df)} rows)")

except Exception as e:
    print(f"\nERROR loading files: {e}")
    sys.exit()

# 3. DEFINE THE RULES
system_prompt = """
You are a smart data assistant capable of reading multiple CSV files.
- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.
- When asked a question, determine which DataFrame is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.
"""

try:
    # ---------------------------------------------------------
    # TODO 1: Initialize the LLM
    # Hint: Use ChatOpenAI with model="gpt-4o-mini" and temperature=0.0
    # ---------------------------------------------------------
    #llm = None # <--- REPLACE 'None' WITH YOUR CODE
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0, api_key=apikey)

    # ---------------------------------------------------------
    # TODO 2: Create the Pandas Agent
    # Hint: Pass the 'llm' and the list 'dataframes' as arguments.
    # Set verbose=True and allow_dangerous_code=True
    # ---------------------------------------------------------
    agent = create_pandas_dataframe_agent(
        llm,
        # <--- PASS THE DATAFRAMES LIST HERE. **Below Line Replaced**
        dataframes,
        verbose=True,
        agent_type="openai-functions",
        allow_dangerous_code=True
    )

    print("\nAI Agent is ready! You can ask questions across ALL files.")
    print("Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'")

except Exception as e:
    print(f"Error initializing agent: {e}")
    sys.exit()

--- Downloading Files from Google Drive ---


Downloading...
From: https://drive.google.com/uc?id=1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq
To: /content/saas_docs.csv
100%|██████████| 2.84k/2.84k [00:00<00:00, 4.19MB/s]


Downloaded: saas_docs.csv


Downloading...
From: https://drive.google.com/uc?id=1_giivc_B0urOKpct0XY2yVZuxW3Eenuf
To: /content/credit_card_terms.csv
100%|██████████| 2.98k/2.98k [00:00<00:00, 5.10MB/s]


Downloaded: credit_card_terms.csv


Downloading...
From: https://drive.google.com/uc?id=1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm
To: /content/hospital_policy.csv
100%|██████████| 3.45k/3.45k [00:00<00:00, 5.83MB/s]


Downloaded: hospital_policy.csv


Downloading...
From: https://drive.google.com/uc?id=1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p
To: /content/ecommerce_faqs.csv
100%|██████████| 3.52k/3.52k [00:00<00:00, 12.5MB/s]


Downloaded: ecommerce_faqs.csv
--- Download Complete ---

ENTER YOUR OPENAI API KEY BELOW:
··········
SUCCESS: Loaded 'saas_docs.csv' (15 rows)
SUCCESS: Loaded 'credit_card_terms.csv' (15 rows)
SUCCESS: Loaded 'hospital_policy.csv' (15 rows)
SUCCESS: Loaded 'ecommerce_faqs.csv' (15 rows)

AI Agent is ready! You can ask questions across ALL files.
Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'


In [ ]:
# ==========================================
#  PART 3: CHAT LOOP
# ==========================================
print("\nType 'exit' or 'quit' to stop conversation.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input:
        continue

    final_query = system_prompt + "\n\nQuestion: " + user_input
    print("AI is thinking...")

    try:
        # ---------------------------------------------------------
        # TODO 4: Invoke the Agent
        # Hint: Use agent.invoke() and pass the final_query
        # The result will be a dictionary, access ['output']
        # ---------------------------------------------------------
        #response = "..." # <--- REPLACE THIS WITH YOUR CODE
        response = agent.invoke({"input": final_query})['output']

        print(f"AI: {response}\n" + "-"*30)
    except Exception as e:
        print(f"An error occurred: {e}")


Type 'exit' or 'quit' to stop conversation.

You: What is your name?
AI is thinking...


> Entering new AgentExecutor chain...
I'm a data assistant here to help you with information from the datasets you have. I don't have a personal name, but you can refer to me as your data assistant! How can I assist you today?

> Finished chain.
AI: I'm a data assistant here to help you with information from the datasets you have. I don't have a personal name, but you can refer to me as your data assistant! How can I assist you today?
------------------------------
You: What is the warranty period on electronics?
AI is thinking...


> Entering new AgentExecutor chain...
The warranty period on electronics is covered under the extended warranty, which includes mechanical and electrical failures after the manufacturer's warranty expires. However, the specific duration of the warranty period is not mentioned in the provided data. You may need to check the manufacturer's details for that information.

